# 10 - Final Demand-Forecasting Models (Consolidated)

This notebook is the organized view of the project's final selected model per series and the resulting 24-month demand forecast (2026-01 to 2027-12). It reads the productionized script outputs as the source of truth and does not re-fit models or re-run model selection.

### Final policy: seven-model open per-target selection

Every target independently compares SARIMA, SARIMAX, Logistic, Gompertz, Ridge, Random Forest, and XGBoost. The winner is chosen only by recursive multi-step walk-forward validation inside the 2023-2024 training window. The 2025 period is an honest holdout report only, never a model-selection input. Pooled regional ML and Diesel Share remain diagnostics only.

| Series | Final model | 2025 holdout MAPE |
|---|---|---:|
| Nacional | SARIMA | 29.0% |
| Madrid | Logistic | 73.6% |
| Cataluña | SARIMAX | 92.3% |
| Andalucía | SARIMA | 49.7% |
| Valencia | Gompertz | 34.2% |


## 0. Setup — load the productionized Phase 2 outputs

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
OUT          = REPO_ROOT / 'data' / 'outputs'
FEAT         = REPO_ROOT / 'data' / 'features'

TARGETS = ['Nacional', 'Madrid', 'Cataluña', 'Andalucía', 'Valencia']
COLORS  = {'Nacional': '#FF6B35', 'Madrid': '#004E89', 'Cataluña': '#1A936F',
           'Andalucía': '#C84B31', 'Valencia': '#8E44AD'}
HEADLINE_FINAL_MODELS = {
    'SARIMA', 'SARIMAX', 'Logistic', 'Gompertz',
    'Ridge', 'Random Forest', 'XGBoost'
}


def reload_production_outputs():
    final_df = pd.read_csv(OUT / 'metricas_final_selected.csv')
    accept_df = pd.read_csv(OUT / 'phase2_model_acceptance.csv')
    pool_exp_df = pd.read_csv(OUT / 'phase2_pooling_experiment_metrics.csv')
    sarima_grid_df = pd.read_csv(OUT / 'sarima_grid_search_results.csv')
    sarima_accept_df = pd.read_csv(OUT / 'sarima_order_acceptance.csv')
    preds_df = pd.read_csv(OUT / 'predicciones_test_2025.csv')
    forecast_df = pd.read_csv(OUT / 'forecast_24m_sarima_rf_xgb.csv')
    selected_forecast_df = pd.read_csv(OUT / 'forecast_24m_selected.csv')
    return final_df, accept_df, pool_exp_df, sarima_grid_df, sarima_accept_df, preds_df, forecast_df, selected_forecast_df

final, accept, pool_exp, sarima_grid, sarima_accept, preds, forecast, selected_forecast = reload_production_outputs()
history  = pd.read_csv(FEAT / 'features_modelo_completo.csv')[['Fecha', 'Target', 'Consumo_Tm']]

SELECTED = dict(zip(final['Target'], final['Model']))
if set(SELECTED.values()) - HEADLINE_FINAL_MODELS:
    print('Detected stale non-headline-eligible selected outputs. Rebuilding production modeling outputs...')
    subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / '05_modeling_with_cnmc.py')], check=True, cwd=REPO_ROOT)
    final, accept, pool_exp, sarima_grid, sarima_accept, preds, forecast, selected_forecast = reload_production_outputs()
    SELECTED = dict(zip(final['Target'], final['Model']))
    if set(SELECTED.values()) - HEADLINE_FINAL_MODELS:
        raise ValueError(f'Final selected models are not all headline-eligible after rebuild: {SELECTED}')

plt.rcParams['figure.dpi'] = 110
print('Final selected model per series:')
for target in TARGETS:
    print(f'  {target}: {SELECTED[target]}')


## 1. Final selected models & how each was chosen

The headline selection and decision trail are loaded from production CSV outputs. `model_selection_walkforward.csv` is the source of the selected model: it ranks the seven headline candidates using only the 2023-2024 recursive walk-forward window. The 2025 metrics in `metricas_final_seleccionado.csv` are reported after selection is fixed.


In [ ]:
# 1a. Final selected models + 2025 validation metrics
tbl = final.copy()
tbl['Pooled'] = np.where(tbl['Model'].str.startswith('Pooled'), 'yes', 'no')
tbl = tbl[['Target', 'Model', 'Pooled', 'MAE', 'RMSE', 'MAPE', 'R2']]
print('FINAL SELECTED FEATURE-AWARE MODELS (2025 validation period):')
print(tbl.to_string(index=False))
avg = final['MAPE'].mean()
print()
print('Average selected 2025 validation MAPE: {:.1f}%'.format(avg))

# 1b. Training-only selection decision per series
print()
print('FEATURE-AWARE SELECTION DECISION (phase2_model_acceptance.csv):')
acc_cols = [c for c in [
    'Target', 'Final_Eligibility_Rule', 'Training_WalkForward_Proposed_Model',
    'Training_WalkForward_Proposed_MAPE', 'Selected_Model',
    'Selected_Model_2025_Validation_MAPE', 'Selected_Uses_Engineered_Features',
    'Final_Selection_Source', 'Decision'
] if c in accept.columns]
print(accept[acc_cols].to_string(index=False))

# 1c. Pooled candidates on the 2025 validation period
print()
print('POOLED CANDIDATES, 2025 validation MAPE (%):')
pe = pool_exp.pivot_table(index='Target', columns='Model', values='MAPE')
cols = [c for c in ['Pooled Ridge', 'Pooled Random Forest', 'Pooled XGBoost'] if c in pe.columns]
print(pe[cols].round(1).to_string())
print()
print('Pooled models are diagnostic only and are not headline-eligible.')


### SARIMA order check

SARIMA parameter tuning is handled in the production script before final model selection. The grid is evaluated only inside the 2023-2024 training period using recursive walk-forward logic, and orders whose training-origin 24-month forecast is degenerate are rejected. There is no 2025 validation / acceptance check for SARIMA orders.


In [ ]:
sarima_selected = (
    sarima_grid[sarima_grid['Selected']]
    [['Target', 'p', 'd', 'q', 'P', 'D', 'Q', 'm', 'WalkForward_MAPE', 'Successful_Folds']]
    .sort_values('Target')
    .reset_index(drop=True)
)
print('TRAINING-ONLY SARIMA GRID WINNERS')
display(sarima_selected)

sarima_accept_cols = [
    'Target', 'Default_Order', 'Default_Seasonal_Order',
    'Grid_Selected_Order', 'Grid_Selected_Seasonal_Order',
    'Grid_WalkForward_MAPE', 'Selected_By_Training_WalkForward',
    'Production_Order', 'Production_Seasonal_Order', 'Decision'
]
print('SARIMA ORDER ACCEPTANCE FOR PRODUCTION')
display(sarima_accept[sarima_accept_cols].sort_values('Target').reset_index(drop=True))


## 2. Results — 2025 holdout fit & the 24-month forecast

First, each series' selected model against the realized 2025 values. Then the deliverable: 2023-2025 history plus the 24-month forward forecast (the dotted grey line marks the train/forecast boundary at 2025-12).

In [ ]:
# 2025 holdout: selected model vs actuals
fig, axes = plt.subplots(3, 2, figsize=(15, 12)); axes = axes.ravel()
for i, t in enumerate(TARGETS):
    ax = axes[i]; m = SELECTED[t]
    d = preds[(preds['Target'] == t) & (preds['Model'] == m)].sort_values('Fecha').copy()
    d['date'] = pd.to_datetime(d['Fecha'])
    ax.plot(d['date'], d['Actual'], 'o-', color='black', lw=2, ms=4, label='Actual')
    ax.plot(d['date'], d['Pred'], 's--', color=COLORS[t], lw=2, ms=4, label=m)
    mape = float(final[final['Target'] == t]['MAPE'].iloc[0])
    ax.set_title('{} - {}  (2025 MAPE {:.1f}%)'.format(t, m, mape), fontsize=11, weight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylabel('Biodiesel (Tm)')
axes[-1].axis('off')
fig.suptitle('2025 holdout: selected model vs actuals', fontsize=13, weight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# 24-month forecast (2026-2027): history + selected-model forecast
fig, axes = plt.subplots(3, 2, figsize=(15, 12)); axes = axes.ravel()
cut = pd.to_datetime('2025-12')
for i, t in enumerate(TARGETS):
    ax = axes[i]; m = SELECTED[t]
    h = history[history['Target'] == t].sort_values('Fecha').copy()
    fc = forecast[(forecast['Target'] == t) & (forecast['Model'] == m)].sort_values('Fecha').copy()
    h['date'] = pd.to_datetime(h['Fecha']); fc['date'] = pd.to_datetime(fc['Fecha'])
    ax.plot(h['date'], h['Consumo_Tm'], '-', color='black', lw=1.8, label='Historical (2023-25)')
    ax.plot(fc['date'], fc['Forecast'], '--', color=COLORS[t], lw=2.2, label='Forecast ({})'.format(m))
    ax.axvline(cut, color='grey', ls=':', lw=1)
    ax.set_title('{} - {}'.format(t, m), fontsize=11, weight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylabel('Biodiesel (Tm)')
axes[-1].axis('off')
fig.suptitle('24-month biodiesel demand forecast (2026-2027)', fontsize=13, weight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# Annual forecast totals (sum of each series' SELECTED model over each year)
f = forecast.merge(pd.DataFrame(SELECTED.items(), columns=['Target', 'Model']), on=['Target', 'Model'])
f['Year'] = f['Fecha'].str[:4]
annual = f.pivot_table(index='Target', columns='Year', values='Forecast', aggfunc='sum').round(0)
annual = annual.reindex(TARGETS)
print('Forecast biodiesel demand (Tm) - annual totals of the per-series selected model:')
print(annual.to_string())

## Summary & caveats

- **What this is:** the consolidated final selected model set from the seven-model open per-target comparison.
- **Selection discipline:** 2025 is an honest holdout report only; it is not used to select model families or SARIMA orders.
- **Honest limits:** only 36 months of data are available and R2 is still negative for every selected target. Present these as **directional planning scenarios**, not precise demand commitments.
- **Tradeoff:** engineered variables are evaluated through SARIMAX and the ML families, but they are not forced to win. SARIMA and growth curves remain valid headline winners when the training-only evidence selects them.
- **Cataluña warning:** SARIMAX wins the training-only comparison by a narrow margin, but its 2025 holdout MAPE is weak and should be disclosed.
